# MicroPlant — Model Compression & Optimization

This notebook focuses on improving model efficiency for edge deployment.

We apply:
- Pruning (remove unnecessary weights)
- Quantization (reduce precision)

Goal:
Achieve a smaller, faster model with minimal performance loss.

In [19]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import torch
import copy
from tqdm.auto import tqdm
tqdm._instances.clear()

from src.architectures import get_microplant
from src.preprocessing import get_dataloaders
from src.training import validate, train_model
from src.compression import apply_global_pruning, remove_pruning_masks, quantize_model
from src.utils import count_parameters, count_model_bytes

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="torch.utils.data.dataloader")

## Setup

Load trained model and dataset.

In [20]:
DATA_DIR = "../data/MicroPlantDataset"
DEVICE = torch.device("cpu")

train_loader, val_loader, test_loader, class_names = get_dataloaders(DATA_DIR)

## Load Trained Model

We load the best model from Notebook 2 (distilled model).

In [21]:
model = get_microplant(num_classes=len(class_names))
model.load_state_dict(torch.load("../models/microplant_kd_best.pth", map_location="cpu"))
model.to(DEVICE)

MicroPlant(
  (stem): Sequential(
    (0): Conv2d(3, 8, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): Hardswish()
  )
  (blocks): Sequential(
    (0): DepthwiseSeparableConv(
      (depthwise): Conv2d(8, 8, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=8, bias=False)
      (bn1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (pointwise): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (se): Sequential(
        (0): AdaptiveAvgPool2d(output_size=1)
        (1): Conv2d(8, 1, kernel_size=(1, 1), stride=(1, 1))
        (2): ReLU(inplace=True)
        (3): Conv2d(1, 8, kernel_size=(1, 1), stride=(1, 1))
        (4): Sigmoid()
      )
      (act): Hardswish()
    )
    (1): DepthwiseSeparableConv(
      (depthwise): Con

## Baseline Metrics

In [22]:
base_loss, base_f1 = validate(model, test_loader, device=DEVICE)

print("Baseline F1:", base_f1)
print("Parameters:", count_parameters(model))
count_model_bytes(model)

Baseline F1: 0.9528606063859291
Parameters: 8804
Model Weights: 35,216 bytes
Model Buffers: 2,648 bytes
Total Size: 36.98 KB


37864

## Global Pruning

We remove less important weights using L1 unstructured pruning.

In [23]:
pruned_model = copy.deepcopy(model)

apply_global_pruning(pruned_model, sparsity=0.5)

Applied global pruning with sparsity 50%


In [24]:
pruned_loss, pruned_f1 = validate(pruned_model, test_loader, device=DEVICE)

print("Pruned F1 (before fine-tune):", pruned_f1)

Pruned F1 (before fine-tune): 0.4632668656615595


### Fine-tuning After Pruning

Pruning damages performance, so we fine-tune the model.

In [26]:
pruned_model = train_model(
    pruned_model,
    train_loader,
    val_loader,
    epochs=10,
    lr=0.0005,
    save_name="../models/pruned",
    device=DEVICE
)

Training: 100%|██████████| 41/41 [00:30<00:00,  1.35it/s]


Epoch 1 | Train Loss 0.3474 F1 0.87 | Val Loss 0.2226 F1 0.9116


Training: 100%|██████████| 41/41 [00:26<00:00,  1.53it/s]


Epoch 2 | Train Loss 0.3062 F1 0.89 | Val Loss 0.2023 F1 0.9088


Training: 100%|██████████| 41/41 [00:25<00:00,  1.59it/s]


...

Epoch 8 | Train Loss 0.2076 F1 0.93 | Val Loss 0.1691 F1 0.9167


Training: 100%|██████████| 41/41 [00:27<00:00,  1.49it/s]


Epoch 9 | Train Loss 0.2003 F1 0.93 | Val Loss 0.1747 F1 0.9241


Training: 100%|██████████| 41/41 [00:26<00:00,  1.57it/s]


Epoch 10 | Train Loss 0.1888 F1 0.94 | Val Loss 0.1732 F1 0.9283


In [27]:
pruned_loss, pruned_f1 = validate(pruned_model, test_loader, device=DEVICE)

print("Pruned F1 (after fine-tune):", pruned_f1)

Pruned F1 (after fine-tune): 0.9501324257667816


In [28]:
remove_pruning_masks(pruned_model)

## Quantization (QAT)

We apply Quantization-Aware Training to further compress the model.

This reduces:
- Model size
- Inference latency

In [29]:
simulated_quant_model, quant_model = quantize_model(
    pruned_model,
    train_loader,
    val_loader,
    epochs=10,
    lr=0.0005,
    save_name="../models/quantized",   
)

c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\ao\quantization\observer.py:534: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  super().__init__(
Training: 100%|██████████| 41/41 [00:43<00:00,  1.07s/it]


Epoch 1 | Train Loss 0.3314 F1 0.89 | Val Loss 0.4000 F1 0.8542


Training: 100%|██████████| 41/41 [00:41<00:00,  1.02s/it]


Epoch 2 | Train Loss 0.3102 F1 0.90 | Val Loss 0.1810 F1 0.9178


Training: 100%|██████████| 41/41 [00:43<00:00,  1.06s/it]


...

Epoch 8 | Train Loss 0.1754 F1 0.94 | Val Loss 0.1682 F1 0.9392


Training: 100%|██████████| 41/41 [00:43<00:00,  1.07s/it]


Epoch 9 | Train Loss 0.1924 F1 0.94 | Val Loss 0.1290 F1 0.9399


Training: 100%|██████████| 41/41 [00:45<00:00,  1.11s/it]


Epoch 10 | Train Loss 0.1756 F1 0.94 | Val Loss 0.1251 F1 0.9577


In [30]:
quant_loss, quant_f1 = validate(simulated_quant_model, test_loader, device='cpu')

print("Quantized F1:", quant_f1)

Quantized F1: 0.9499222519006844


## Model Comparison

In [31]:
print("=== Results ===")
print(f"Baseline F1:  {base_f1:.4f}")
print(f"Pruned F1:    {pruned_f1:.4f}")
print(f"Quantized F1: {quant_f1:.4f}")

=== Results ===
Baseline F1:  0.9529
Pruned F1:    0.9501
Quantized F1: 0.9499


In [32]:
print("\n=== Size Comparison ===")
print("Baseline:")
count_model_bytes(model)

print("\nPruned:")
count_model_bytes(pruned_model)

print("\nQuantized:")
count_model_bytes(quant_model)


=== Size Comparison ===
Baseline:
Model Weights: 35,216 bytes
Model Buffers: 2,648 bytes
Total Size: 36.98 KB

Pruned:
Model Weights: 35,216 bytes
Model Buffers: 2,648 bytes
Total Size: 36.98 KB

Quantized:
Model Weights: 2,560 bytes
Model Buffers: 2,808 bytes
Total Size: 5.24 KB


5368

## Observations

- Pruning reduces model complexity with slight accuracy drop
- Fine-tuning helps recover performance
- Quantization significantly reduces size with minimal loss

### Conclusion
The MicroPlant model achieves strong efficiency-performance tradeoff, making it suitable for deployment on edge devices.